# ACV Bolivia – Ejemplo completo con ACVEngine

Pipeline: **1) build → 2) LCA → 3) Monte Carlo → 4) Sensibilidad → 5) Reportes**.

Cada fase guarda sus resultados en disco (`save_cache`, por defecto `True`) y con
`use_cache=True` los reutiliza entre sesiones sin recalcular ni exigir la fase anterior.
Los gráficos se muestran aquí mismo (inline) y se guardan como PNG cuando procede.

## 0. Preparación

Aplicamos un parche a `SciPy` (`atol='legacy'`) recurrente en los solvers internos de Brightway2.

In [ ]:
import functools

import scipy.sparse.linalg as spla


def clean_scipy_args(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if kwargs.get("atol") == "legacy":
            kwargs.pop("atol")
        return func(*args, **kwargs)

    return wrapper


spla.cgs = clean_scipy_args(spla.cgs)
spla.cg = clean_scipy_args(spla.cg)
spla.bicgstab = clean_scipy_args(spla.bicgstab)
spla.gmres = clean_scipy_args(spla.gmres)
print("SciPy monkey-patch aplicado globalmente.")

## 0.1. Configuración y motor de cálculo

Cargamos la configuración base (`settings.json`) y creamos `eng`, un `ACVEngine`.

In [ ]:
from acv_bolivia import ACVEngine, AppConfig

path_settings = "C:\\Users\\sandr\\OneDrive\\Escritorio\\TFM\\VS_Code\\otras_pruebas\\ACVBolivia\\settings.json"
config = AppConfig.load_from_json(path_settings)
eng = ACVEngine(config)  # o tambien: eng = ACVEngine.from_json(path_settings)

---

## 1. Construir el Inventario

Carga los parámetros desde el Excel y aplica los enlaces con Brightway2-Ecoinvent:
* Inventario de masas por componente.
* Mapeo de procesos de Ecoinvent (con código y unidad resueltos).
* Dependencias y reglas entre componentes.
* Incertidumbre por componente y PDF.

In [ ]:
build_inventory = eng.build(
    force_rebuild = False, # True: reconstruir el inventario desde cero.
)

---

## 2. Cálculo LCIA (LCA)

Analiza el inventario de procesos, itera sobre cada método de impacto y componente, y permite
extraer impactos por componente o el total sobre **n** hotspots (componentes de mayor impacto).

In [ ]:
calculate_lca = eng.run_lca(
    patron_metodo = "", # ReCiPe, CED, etc.
    nivel_metodo = "", # midpoint (H), endpoint(H) or (E), etc.
    top_n_hotspot = "", # componentes registrados por método.
    functional_unit = "", # Unidad Funcional, default = 1.0.
    generation_dict = "", # Energia total para normalización por kWh.
    use_cache = True,
    save_cache = True,
    cache_filename = "LCA_ED", # None = 'lca_results'; p. ej. 'lca_delimitadora'.
)

Identificamos el proyecto y el método que usaremos en todo el ejemplo.

In [ ]:
project_name = eng.build_result.project_names # list[str]
p: int = 0 # Índice del proyecto para el análisis posterior.

lca_result = eng.lca_result
metodos = lca_result.methods # tuple('patron, nivel', 'metodo', 'metodo_')
m: int = 4 # Índice del método para el análisis posterior.

print(f"Proyecto: '{project_name[p]}'")
print(f"Método {m}: {metodos[m]}")

### 2.1. Gráficos LCA

Comparativa entre proyectos (normalizada por kWh).

In [ ]:
fig, ax = eng.plotter.graficar_comparativa(
    usar_kwh = True, # comparación relativa.
    top_n = None, # métodos a comparar
)

Hotspots apilados (contribución por proceso del proyecto).

In [ ]:
fig, ax = eng.plotter.graficar_hotspot_apilados(
    project_name = project_name[p],
    top_n = None, # cantidad de componentes (hotspots) por método.
)

---

## 3. Simulación de Monte Carlo (SMC)

Tres modos combinables (ver `docs/METODOLOGIA.md`):

* `run_bw_mc=True` – Monte Carlo completo de Brightway2 (foreground + background).
* `run_foreground_mc=True` – solo foreground (Excel).
* `run_piv=True` – aproximación lineal PIV (h-vectors × muestras), con o sin pedigrí.

Cada escenario se guarda bajo `BASE_OUTPUT/<fase>/<fecha>/<cache_filename>.pkl.gz` y se
recupera en otra sesión con `use_cache=True`.

> Nota: el escenario **PIV** también alimenta con sus muestras a los analizadores de
> sensibilidad basados en muestras (correlation/PRCC, regression/SRRC y SHAP). Ejecutar
> PIV antes de `run_sensitivity()` habilita esos métodos sin un MC foreground aparte.

### 3.1. Monte Carlo completo (Brightway2)

In [ ]:
smc = eng.run_montecarlo(
    run_bw_mc = True, # SMC completo FG+BG
    run_foreground_mc = False, # SMC fg
    run_piv = False, # MC PIV
    iterations = 1000, # para MC completo
    fg_iterations = 1_000, # para fg y PIV
    fg_seed = 42, # semilla para reproducibilidad
    include_pedigree = False, # variabilidad (muestreo)
    correlate_pedigree = False, # correlación física entre componentes
    enforce_physical_constraints = False, # truncar resultados negativos / solo para piv
    verbose_processor = True, # logs del proceso
    use_cache = True,
    save_cache = True,
    cache_filename = "ED_SMC_Full1000", # None = 'montecarlo_results'.
)

#### 3.1.1. Gráficos del MC completo

In [ ]:
fig, ax = eng.plotter.graficar_mc_distribucion(
    project_name = project_name[p],
    method_id = metodos[m],
    bins = 100,
)

In [ ]:
fig, ax = eng.plotter.graficar_mc_boxplots(method_id = metodos[m])

### 3.2. Monte Carlo PIV (aproximación lineal)

El modo PIV produce, además de los scores, las **contribuciones por componente** necesarias
para los plotters PIV y para los analizadores de sensibilidad por muestra.

> Al estar ya disponible, `eng.piv_plotter()` se puede usar en esta fase.

In [ ]:
smc_piv = eng.run_montecarlo(
    run_bw_mc = False,
    run_foreground_mc = False,
    run_piv = True, # aproximación lineal PIV
    iterations = 1000,
    fg_iterations = 1_000, # muestras PIV
    fg_seed = 42, # semilla para reproducibilidad
    include_pedigree = False,
    correlate_pedigree = False,
    enforce_physical_constraints = False,
    use_cache = True,
    save_cache = True,
    cache_filename = "ED_SMC_PIV1000", # None = 'montecarlo_results'.
)

#### 3.2.1. Gráficos PIV (requiere modo PIV en el motor)

In [ ]:
piv_plotter = eng.piv_plotter(project_id = project_name[p])
piv_plotter.piv_hotspot_distributions(
    project_name = project_name[p],
    method_id = metodos[m],
)

### 3.3. Convergencia del Monte Carlo

`eng.run_convergence_diagnostics(proyecto, método)` ejecuta **5 chequeos** y devuelve un `ConvergenceReport`:
1. **Media y CV acumulados** – la media se estabiliza dentro de la tolerancia (def. 2%).
2. **Error estándar de Monte Carlo** – precisión relativa frente a objetivos (1%, 2%, 5%).
3. **Comparación de mitades** – 1ª mitad vs 2ª mitad de la serie (d de Cohen < 0.2).
4. **Percentiles** – p5-p95 estabilizados antes del 90% de N.
5. **Entre semillas** – consistencia entre corridas independientes (con `seed_comparison`).

Comparamos **dos corridas de N=1000 con semillas distintas** (`smc_v1000A_seed` y `smc_v1000B_seed`):
la primera ejecución calcula y guarda; las siguientes cargan desde caché al instante.

In [ ]:
# Escenario A: N=1000, semilla 42
eng.run_montecarlo(
    run_bw_mc = True,
    run_piv = False,
    run_foreground_mc = False,
    iterations = 1000,
    fg_seed = 42,
    cache_filename = "smc_v1000A_seed",
)
scores_v1000A = eng.mc_result.get_scores(metodos[m], project_name[p])

# Escenario B: N=1000, semilla 98
eng.run_montecarlo(
    run_bw_mc = True,
    run_piv = False,
    run_foreground_mc = False,
    iterations = 1000,
    fg_seed = 98,
    cache_filename = "smc_v1000B_seed",
)
scores_v1000B = eng.mc_result.get_scores(metodos[m], project_name[p])

print(f"v1000A: {len(scores_v1000A)} muestras  |  v1000B: {len(scores_v1000B)} muestras")

Diagnóstico de convergencia por versión (la caché evita recalcular).

In [ ]:
# Versión B (N=1000), la que quedó cargada en el motor:
diag_v1000B = eng.run_convergence_diagnostics(project_name[p], metodos[m][1])
print(diag_v1000B.summary())

# Versión A (N=1000): cargar desde caché y diagnosticar
eng.run_montecarlo(cache_filename = "smc_v1000A_seed")
diag_v1000A = eng.run_convergence_diagnostics(project_name[p], metodos[m][1])
print(diag_v1000A.summary())

print("\nVEREDICTO A (semilla 42):", diag_v1000A.is_converged)
print("VEREDICTO B (semilla 98):", diag_v1000B.is_converged)

¿Coinciden las dos semillas? Si las distribuciones son consistentes (d de Cohen < 0.2),
la simulación ya estabilizó en N=1000.

In [ ]:
from acv_bolivia.analysis.convergence import seed_comparison

comp = seed_comparison(
    [scores_v1000A, scores_v1000B],
    labels = ["N=1000A", "N=1000B"],
    method_name = metodos[m][1],
)
print(comp.summary())

---

## 4. Análisis de Sensibilidad

Multi-método: **Delta LCA**, **Morris**, **Sobol**, **Correlation/PRCC**, **Regression/SRRC** y **SHAP**.

`delta_lca`, `morris` y `sobol` evalúan el modelo **en vivo** (sin simulación previa).
`correlation`, `regression` y `shap` consumen las muestras del escenario **PIV** ya ejecutado
(no requieren un MC foreground aparte). `run_sensitivity()` solo calcula los reportes;
los gráficos y el Excel se generan con `plot_sensitivity()` y `export_sensitivity()`.

In [ ]:
sensitivity_analisys = eng.run_sensitivity(
    analyzers = None, # Ejecuta todos los analizadores
    exclude_methods = {"morris", "sobol"}, # habilitarlos quita el comentario
    method_indices = [m], # método a evaluar
    project_indices = None, # proyectos a evaluar
    use_cache = True,
    save_cache = True,
    cache_filename = "ED_sensibilidad", # None = 'sensitivity_results'.
)
rep = eng.sensitivity_result.get_report(project_name[p], metodos[m])
print(f"Métodos ejecutados: {rep.methods_run}  |  errores = {rep.has_errors}")

### 4.1. Gráficos de sensibilidad

Con `SensitivityPlotter` mostramos cada gráfico inline; `eng.plot_sensitivity()` los guarda
además como PNG (con `close_figs=False` quedan también visibles).

In [ ]:
from acv_bolivia.analysis.sensitivity import SensitivityPlotter

sp = SensitivityPlotter()
fig, ax = sp.plot_delta(rep, top_n = 10) # diagrama de tornado · Delta LCA

In [ ]:
fig, ax = sp.plot_shap(rep, top_n = 15) # beeswarm SHAP

In [ ]:
fig, ax = sp.plot_correlation(rep, top_n = 6) # scatter de correlación

Guardamos todos los gráficos aplicables como PNG (con las figuras visibles).

In [ ]:
eng.plot_sensitivity(
    project_id = project_name[p],
    method_id = metodos[m],
    close_figs = False, # True: guarda y cierra; False: guarda y deja visibles.
)

### 4.2. Validación SHAP vs PIV

Compara la importancia SHAP con la contribución lineal PIV por componente; si el LCA es
lineal, los puntos caen sobre la diagonal.

In [ ]:
piv_plotter.shap_vs_piv_scatter(
    report = rep,
    project_name = project_name[p],
    method_id = metodos[m],
)

In [ ]:
piv_plotter.plot_all_piv(
    report = rep,
    project_name = project_name[p],
    method_id = metodos[m],
    top_n = 8,
)

### 4.3. Confiabilidad y estabilidad de la sensibilidad

El paquete `acv_bolivia.analysis.convergence` ofrece:
* **Confiabilidad de índices** – `summarize_sobol_reliability()` / `summarize_morris_reliability()`
  (requieren haber habilitado `sobol`/`morris` en `run_sensitivity`).
* **Estabilidad del ranking** – `ranking_stability(rankings, labels, top_k)`: orden de importancia
  entre versiones (Spearman ρ + solapamiento del top-k). Ideal para comparar `n_synthetic_samples`.

Generamos dos versiones con distinto `n_synthetic_samples` (`sens_v200` y `sens_v500`).

In [ ]:
# Versión A: n_synthetic_samples = 200
eng.run_sensitivity(
    exclude_methods = {"sobol", "morris"},
    method_indices = [m],
    n_synthetic_samples = 200,
    cache_filename = "sens_v200",
)
rep_v200 = eng.sensitivity_result.get_report(project_name[p], metodos[m])

# Versión B: n_synthetic_samples = 500
eng.run_sensitivity(
    exclude_methods = {"sobol", "morris"},
    method_indices = [m],
    n_synthetic_samples = 500,
    cache_filename = "sens_v500",
)
rep_v500 = eng.sensitivity_result.get_report(project_name[p], metodos[m])

for etiqueta, rep_ in [("sens_v200", rep_v200), ("sens_v500", rep_v500)]:
    print(f"{etiqueta}: métodos ejecutados = {rep_.methods_run}  |  errores = {rep_.has_errors}")

In [ ]:
from acv_bolivia.analysis.convergence import (
    summarize_morris_reliability,
    summarize_sobol_reliability,
)

if "morris" in rep_v500.results:
    print(summarize_morris_reliability(rep_v500.get_raw("morris")).summary())
if "sobol" in rep_v500.results:
    print(summarize_sobol_reliability(rep_v500.get_raw("sobol")).summary())

print("\nTop componentes (sens_v500):", rep_v500.top_components(n = 10))

In [ ]:
from acv_bolivia.analysis.convergence import ranking_stability


def ranking_delta(rep_):
    res = rep_.results["delta_lca"]
    return [s.component for s in sorted(res.scores, key=lambda x: abs(x.score), reverse=True)]


estabilidad = ranking_stability(
    [ranking_delta(rep_v200), ranking_delta(rep_v500)],
    labels = ["sens_v200", "sens_v500"],
    top_k = 5,
)
print(estabilidad.summary())

---

## 5. Reportes

Exporta los resultados LCIA/MC y de sensibilidad a Excel, y lista las cachés de cada fase.

In [ ]:
eng.export("ACV_Reporte_Final")
eng.export_sensitivity(
    project_id = project_name[p],
    method_id = metodos[m],
    nombre = "Sensibilidad_Reporte_Final",
)
print(eng.list_caches("lca"))
print(eng.list_caches("montecarlo"))
print(eng.list_caches("sensibilidad"))

---
**Fin del pipeline completo.**

| Para | Haz | Dónde |
|---|---|---|
| Recalcular desde cero | `use_cache=False` por escenario, o borrar `BASE_OUTPUT` | en cada `run_*` |
| Ver escenarios guardados | `eng.list_caches('lca'/'montecarlo'/'sensibilidad')` | §5 |
| Convergencia Monte Carlo | `eng.run_convergence_diagnostics(proyecto, método)` | §3.3 |
| Confiabilidad sensibilidad | `summarize_*_reliability` / `ranking_stability` | §4.3 |
| Ver gráficos inline | `SensitivityPlotter().plot_*` o `plot_sensitivity(close_figs=False)` | §4.1 |
| Documentación completa | `docs/MANUAL_ACVENGINE.md` y `dev/GUIA_CONTROL_FRAMEWORK.md` | repo |